Prácticas del Curso de Modelado y Simulación Topológica, Álvaro Torras Casas (c) 2026, 
Departamento de Matemática Aplicada I, 
Escuela Técnica Superior de Ingeniería Informática, Universidad de Sevilla, España.

# Práctica 1: Homología Simplicial

Vamos a estudiar como calcular homología simplicial sobre un cuerpo mediante algunas librerías de Python. 

Empezamos instalando los módulos necesarios. 
Sobretodo, vamos a tener que usar la librería `Numpy` (https://numpy.org/) de python que sirve para trabajar con matrices.
Por otro lado, para los cálculos con complejos simpliciales y homología necesitaremos utilizar la librería `GUDHI` (https://gudhi.inria.fr/) 

In [ ]:
%%capture
pip install numpy gudhi

También utilizaremos las liberías `NetworkX` y `Matplotlib` para visualizar algunos complejos simpliciales.

In [ ]:
%%capture
pip install networkx matplotlib

Por otro lado, vamos a instalar el módulo PHAT (https://www.sciencedirect.com/science/article/pii/S0747717116300098) que nos permitirá trabajar con las matrices sobre $\mathbb{Z}_2$. Para esto utilizaremos una versión algo modificada.

In [ ]:
%%capture
!pip install setuptools pybind11
!pip install --no-build-isolation git+https://bitbucket.org/atorras1618/phat.git

Como seguramente estaremos ejecutando la libreta en Google Colab, ejecutamos la siguiente celda para importar funciones auxiliares para trabajar con los laboratorios.

In [ ]:
%%capture
import os

# Check if we are running in Colab
if 'google.colab' in str(get_ipython()):
    repo_name = 'MyST'
    if not os.path.exists(repo_name):
        !git clone https://github.com/atorras1618/{repo_name}.git
    
    os.chdir(repo_name)
    import sys
    sys.path.append(os.getcwd())

import funciones_auxiliares

# Parte 1: manipulaciones básicas de complejos simpliciales abstractos

En esta parte trabajaremos con el objeto simplex Tree de Gudhi para definir algunos complejos simpliciales abstractos.

### Ejemplo 1: Un complejo sencillo
Vamos a crear un complejo simplicial mediante la estructura de datos `simplex_tree` de GUDHI. Para esto, importaremos el objeto simplex tree con el alias `st` y le añadiremos los símplices del siguiente complejos simplicial $K$:

![Ejemplo 1](images/ejemplo-1.png)

In [ ]:
import gudhi
K = gudhi.SimplexTree()
# Añadimos los símplices de dimensión 0
K.insert([0])
K.insert([1])
K.insert([2])
K.insert([3])
# Añadimos los símplices de dimensión 1
K.insert([0,1])
K.insert([0,2])
K.insert([1,2])
K.insert([1,3])
K.insert([2,3])
# Añadimos el símplice de dimensión 2
K.insert([1,2,3])

Cada vez que insertamos un símplice en el objeto `st`, el método devuelve `True` cuando la inserción se ha realizado con éxito. Por otro lado, podemos comprobar leer todos los símplices que hemos insertado mediante el método `get_simplices()`.

In [ ]:
print(list(K.get_simplices()))

Como podemos combrobar, la lista que nos devuelve GUDHI incluye los valores de filtración de todos los símplices. Como no los hemos configurado, todos toman el valor por defecto `0.0`. En estas prácticas incluimos dos funciones para trabajar con la lista de símplices e imprimirla en pantalla: `diccionario_simplices(K)`y `ver_simplices(K)`



In [ ]:
from funciones_auxiliares import diccionario_simplices, ver_simplices

In [ ]:
dict_spx = diccionario_simplices(K)
for dim in range(3):
    print(f"Símplices en dimensión {dim}:")
    print(dict_spx[dim])

El mismo resultado lo podemos obtener de forma más cómoda mediante `ver_simplices`

In [ ]:
ver_simplices(K)

Vamos ahora a representar el complejo simplicial en el plano. 

In [ ]:
from funciones_auxiliares import plot_simplex_tree_2D

plot_simplex_tree_2D(K, figsize=(4,4))

Podemos asignar posiciones a los nodos y especificarlas a la hora de representar el complejo simplicial:

In [ ]:
pos={0:[0,0],1:[1,0],2:[0,1],3:[1,1]}
plot_simplex_tree_2D(K, pos=pos, figsize=(4,4))

Alternativamente, podemos crear el mismo complejo simplicial añadiendo únicamente los símplices maximales, sin necesidad de añadir sus caras de forma explícita.

In [ ]:
import gudhi
K_aux = gudhi.SimplexTree()
# Añadimos los símplices maximales 
# de dimensión 1
K_aux.insert([0,1])
K_aux.insert([0,2])
# y el de dimensión 2
K_aux.insert([1,2,3])

Y comprobamos que el complejo simplicial resultante es idéntico al anterior.

In [ ]:
ver_simplices(K_aux)
plot_simplex_tree_2D(K_aux, pos=pos, figsize=(4,4))

Nótese que Gudhi guarda los símplices como pares (símplice, valor de filtración). El valor por defecto es `0.0`.

In [ ]:
print(f"El complejo simplicial tiene {K.num_vertices()} vértices y un total de {K.num_simplices()} símplices y su dimensión es {K.dimension()}.")

Podemos calcular la característica de Euler $\chi(K)$ sumando y restando símplices, en este caso obtenemos $\chi(K)=0$

In [ ]:
dict_spx = diccionario_simplices(K)
euler_chi = 0
for dim in range(K.dimension()+1):
    euler_chi += ((-1)**dim) * len(dict_spx[dim])

print(f"La característica de Euler del complejo simplicial es {euler_chi}")

Por comodidad, esta misma función se encuentra en el fichero de funciones auxiliares, lo podemos cargar y ejecutar:

In [ ]:
from funciones_auxiliares import característica_euler

característica_euler(K)

Gudhi también permite calcular los números de Betti del complejo simplicial. Para esto, simplimente ejecutaremos el método `compute_persistence()`seguido de la función que nos devuelve los números de Betti.

In [ ]:
K.compute_persistence()
numeros_Betti = K.betti_numbers()
print(numeros_Betti)

Entonces obtenemos $\beta_0(K)=1$ y $\beta_1(K)=1$, cumpliéndose la fórmula $\chi(K)=\beta_0(K)-\beta_1(K)=0$ (en este caso los números de Betti de dimensiiones $\geq 2$ son ignorados pur GUDHI. Si queremos obtener números de Betti superiores, por ejemplo, hasta dimensión 5, entonces debemos decir a GUDHI que el complejo simplicial tiene dimensión $6$:

In [ ]:
K.set_dimension(6)
K.compute_persistence()
numeros_Betti = K.betti_numbers()
print(numeros_Betti)

### Ejemplo 2: Triangulación de un tetraedro vacío

Consideramos un tetraedro vacío al que llamaremos `T`. Este lo podemos definir fácilmente, incluyendo primero todo el tetraedro y, seguidamente, substrayendo el símplice maximal mediante el método `remove_maximal_simplex`

In [ ]:
T = gudhi.SimplexTree()
T.insert([0,1,2,3])
T.remove_maximal_simplex([0,1,2,3])

Podemos comprovar que efectivamente `T`es el tetraedro vacío. Nótese que la representación tiene sus limitaciones.

In [ ]:
ver_simplices(T)
plot_simplex_tree_2D(T)

Podemos inspeccionar algunas propiedades de `T`:

In [ ]:
print(f"Número de vértices en T: {T.num_vertices()}")
print(f"Número de símplices en T: {T.num_simplices()}")
print(f"Dimensión de T: {T.dimension()}")
print(f"Característica de Euler de T: {característica_euler(T)}")

Cuando calculamos los números de Betti, vemos que algo no acaba de encajar:

In [ ]:
T.compute_persistence()
print(f"Betti numbers: {T.betti_numbers()}")

En particular, tenemos que $\chi(T)=2 \neq 1 = \beta_0(T) - \beta_1(T)$. En realidad---como ya hemos comentado en el ejemplo anterior---debemos indicar a GUDHI que el complejo con el que estamos tratando tiene una dimensión más. Esto asegura el cálculo de los números de Betti hasta dimensión $2$:

In [ ]:
T.set_dimension(3)
T.compute_persistence()
print(f"Betti numbers: {T.betti_numbers()}")

De esta forma, se comprueba la fórmula conocida: $\chi(T)=\beta_0(T) - \beta_1(T) + \beta_2(T)$

### Ejemplo 3: un toro

Vamos a considerar una triangulación cásica del toro `Toro` que consiste en 16 triángulos como en la siguente figura:

![Ejemplo 1](images/toro.png)

In [ ]:
Toro = gudhi.SimplexTree()

# Define triangles for a torus triangulation
triangles = [
    [0,1,3], [1,3,4], [1,2,4], [2,4,5], [0,2,5], [0,3,5], # primera columna
    [3,4,6], [4,6,7], [4,5,7], [5,7,8], [3,5,8], [3,6,8], # segunda columna
    [0,6,7], [0,1,7], [1,7,8], [1,2,8], [2,6,8], [0,2,6]  # tercera columna
]

# Insert simplices into the tree
for triangle in triangles:
    Toro.insert(triangle)

In [ ]:
ver_simplices(Toro)

Podemos entonces imprimir información relativa al toro. Esta vez ya incluimos que la dimensión debe ser $3$.

In [ ]:
print(f"Number of vertices: {Toro.num_vertices()}")
print(f"Number of simplices: {Toro.num_simplices()}")
Toro.set_dimension(3)
print(f"Dimension: {Toro.dimension()}")

# Compute persistence to verify the Betti numbers
# For a torus, we expect: B0=1, B1=2, B2=1
Toro.compute_persistence()
betti = Toro.betti_numbers()
print(f"Betti numbers: {betti}")

## Parte 2: cálculos matriciales y representantes de clases de homología

